In [1]:
#importing libraries and setting the working file directory
import cv2
import os 
import numpy as np 
import matplotlib.pyplot as plt

print("Your working folder is:", os.getcwd())

output_folder = "frames/raw"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(" RAW folder created successfully!")
else:
    print(" RAW folder already exists!")

print("Saving frames in:", os.path.abspath(output_folder))


#extracting frames

video_path = "cross.mov"   

if not os.path.exists(video_path):
    print(" Video file not found! Check the file name.")
    print(" Your current folder is:", os.getcwd())

else:
    cap = cv2.VideoCapture(video_path)

    frame_count = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if frame_count % 10 == 0:
            file_path = os.path.join(output_folder, f"frame_{saved_count:04d}.jpg")
            cv2.imwrite(file_path, frame)
            saved_count += 1

        frame_count += 1

    cap.release()

    print("Extraction done")
    print("Total Frames Saved:", saved_count)
    print("Saved in:", os.path.abspath(output_folder))


grayscale_folder = "frames/grayscale"

if not os.path.exists(grayscale_folder):
    os.makedirs(grayscale_folder)
    print("Grayscale folder created successfully!")
else:
    print(" Grayscale folder already exists!")

print("Grayscale frames will be saved in:", os.path.abspath(grayscale_folder))


frame_files = sorted(os.listdir(output_folder))

converted_count = 0

for fname in frame_files:
    img = cv2.imread(os.path.join(output_folder, fname))

    if img is None:
        print(f" Could not load: {fname}")
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


    save_path = os.path.join(grayscale_folder, fname)
    cv2.imwrite(save_path, gray)

    plt.figure()
    plt.title(f"Histogram of {fname}")
    plt.xlabel("Pixel Intensity")
    plt.ylabel("Frequency")
    plt.hist(gray.ravel(), bins=256, range=(0, 256))
    
    hist_path = os.path.join(grayscale_folder, f"{os.path.splitext(fname)[0]}_hist.png")
    plt.savefig(hist_path)
    plt.close()

    converted_count += 1

print(" Grayscale Conversion Completed!")
print(" Total Frames Converted:", converted_count)
print(" Saved in:", os.path.abspath(grayscale_folder))

selected_frames = [
    "frame_0017.jpg",
    "frame_0018.jpg",
    "frame_0019.jpg",
    "frame_0020.jpg"
]

for fname in selected_frames:
    img_path = os.path.join(grayscale_folder, fname)
    gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if gray is None:
        print("Could not load:", fname)
        continue

enhanced_folder = "frames/enhanced"

if not os.path.exists(enhanced_folder):
    os.makedirs(enhanced_folder)
    print("Enhanced folder created!")
else:
    print("Enhanced folder already exists!")

print("Enhanced images will be saved in:", os.path.abspath(enhanced_folder))

median_folder = "frames/enhanced/median"
if not os.path.exists(median_folder):
    os.makedirs(median_folder)
    print("Median folder created!")
else:
    print("Median folder already exists!")

print("Median results:", os.path.abspath(median_folder))

def calculate_entropy(image):
    histogram, _ = np.histogram(image.flatten(), bins=256, range=(0,256))
    histogram = histogram / histogram.sum()
    histogram = histogram[histogram > 0]
    entropy = -np.sum(histogram * np.log2(histogram))
    return entropy


def calculate_psnr(original, enhanced):
    mse = np.mean((original - enhanced) ** 2)
    if mse == 0:
        return 100
    return 20 * np.log10(255.0 / np.sqrt(mse))


# ------Median Filtering on Selected Frames 
median_results = []  

for fname in selected_frames:
    gray_path = os.path.join(grayscale_folder, fname)
    gray = cv2.imread(gray_path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        continue

   
    median = cv2.medianBlur(gray, 17) 

  
    save_path = os.path.join(median_folder, fname)
    cv2.imwrite(save_path, median)


    mean_before = np.mean(gray)
    mean_after = np.mean(median)

    std_before = np.std(gray)
    std_after = np.std(median)

    entropy_before = calculate_entropy(gray)
    entropy_after = calculate_entropy(median)

    psnr_value = calculate_psnr(gray, median)

    median_results.append([
        fname,
        mean_before,
        mean_after,
        std_before,
        std_after,
        entropy_before,
        entropy_after,
        psnr_value
    ])

import pandas as pd

columns = ["Frame", "Mean Before", "Mean After", "Std Before", "Std After", "Entropy Before", "Entropy After", "PSNR"]
df_median = pd.DataFrame(median_results, columns=columns)

df_median

df_median.to_csv("frames/enhanced/median/median_metrics.csv", index=False)

clahe_folder = "frames/enhanced/clahe"

selected_frames = [
    "frame_0017.jpg",
    "frame_0018.jpg",
    "frame_0019.jpg",
    "frame_0020.jpg"
]

if not os.path.exists(clahe_folder):
    os.makedirs(clahe_folder)
    print("CLAHE folder created!")
else:
    print("CLAHE folder already exists!")

clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
clahe_results = []

for fname in selected_frames:
    gray_path = os.path.join(grayscale_folder, fname)
    img = cv2.imread(gray_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        print("Could not load:", fname)
        continue


    clahe_img = clahe_obj.apply(img)


    save_path = os.path.join(clahe_folder, fname)
    cv2.imwrite(save_path, clahe_img)

   
 
    mean_before = np.mean(img)
    mean_after = np.mean(clahe_img)
    std_before = np.std(img)
    std_after = np.std(clahe_img)
    entropy_before = calculate_entropy(img)
    entropy_after = calculate_entropy(clahe_img)
    psnr_value = calculate_psnr(img, clahe_img)

    clahe_results.append([
        fname,
        mean_before,
        mean_after,
        std_before,
        std_after,
        entropy_before,
        entropy_after,
        psnr_value
    ])


import pandas as pd

columns = ["Frame", "Mean Before", "Mean After", "Std Before", "Std After", "Entropy Before", "Entropy After", "PSNR"]
df_clahe = pd.DataFrame(clahe_results, columns=columns)
df_clahe

df_clahe.to_csv("frames/enhanced/clahe/clahe_metrics.csv", index=False)

# ---------------- LAPACIAN EDGE TEST + SHARPENING ----------------

laplacian_folder = "frames/enhanced/laplacian"

if not os.path.exists(laplacian_folder):
    os.makedirs(laplacian_folder)
    print("Laplacian folder created!")
else:
    print("Laplacian folder already exists!")

laplacian_results = []

for fname in selected_frames:

 
    clahe_path = os.path.join(clahe_folder, fname)
    img = cv2.imread(clahe_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        continue

 
    lap = cv2.Laplacian(img, cv2.CV_64F)
    lap_abs = np.abs(lap)

   
    mid_row = lap_abs[lap_abs.shape[0]//2, :]



    save_graph_before = os.path.join(laplacian_folder, fname.replace(".jpg","_lap_graph_before.png"))
    plt.figure(figsize=(10,4))
    plt.plot(mid_row)
    plt.title(f"Laplacian Graph Before - {fname}")
    plt.xlabel("Pixel Position")
    plt.ylabel("Laplacian Value")
    plt.savefig(save_graph_before)
    plt.close()


    lap_mean = np.mean(lap_abs)

    if lap_mean < 15:


        lap_uint8 = cv2.convertScaleAbs(lap)

        lap_sharp = cv2.addWeighted(img, 0.5, lap_uint8, -0.1, 0)

      
        lap_new = cv2.Laplacian(lap_sharp, cv2.CV_64F)
        lap_new_abs = np.abs(lap_new)

        mid_row_new = lap_new_abs[lap_new_abs.shape[0]//2, :]
    
    
        save_img_path = os.path.join(laplacian_folder, fname.replace(".jpg","_lap_enhanced.jpg"))
        cv2.imwrite(save_img_path, lap_sharp)

      
        save_graph_after = os.path.join(laplacian_folder, fname.replace(".jpg","_lap_graph_after.png"))
        plt.figure(figsize=(10,4))
        plt.plot(mid_row_new)
        plt.title(f"Enhanced Laplacian Graph - {fname}")
        plt.xlabel("Pixel Position")
        plt.ylabel("Laplacian Value")
        plt.savefig(save_graph_after)
        plt.close()

        
        mean_before = np.mean(img)
        mean_after = np.mean(lap_sharp)

        std_before = np.std(img)
        std_after = np.std(lap_sharp)

        entropy_before = calculate_entropy(img)
        entropy_after = calculate_entropy(lap_sharp)

        psnr_value = calculate_psnr(img, lap_sharp)

        laplacian_results.append([
            fname,
            mean_before, mean_after,
            std_before, std_after,
            entropy_before, entropy_after,
            psnr_value
        ])

    else:
        print(f"{fname} already has strong edges, skipping sharpening.")



columns = [
    "Frame",
    "Mean Before","Mean After",
    "Std Before","Std After",
    "Entropy Before","Entropy After",
    "PSNR"
]

df_laplacian = pd.DataFrame(laplacian_results, columns=columns)

csv_path = os.path.join(laplacian_folder, "laplacian_metrics.csv")
df_laplacian.to_csv(csv_path, index=False)

print("\nLaplacian enhancement completed!")
print("Results saved in:", laplacian_folder)


display(df_laplacian)

df_laplacian.to_csv("frames/enhanced/laplacian/laplacian_metrics.csv", index=False)





Your working folder is: C:\Users\Mavith\OneDrive\Desktop\ip_project\Cross_marking_removel\pipeline
 RAW folder created successfully!
Saving frames in: C:\Users\Mavith\OneDrive\Desktop\ip_project\Cross_marking_removel\pipeline\frames\raw
Extraction done
Total Frames Saved: 31
Saved in: C:\Users\Mavith\OneDrive\Desktop\ip_project\Cross_marking_removel\pipeline\frames\raw
Grayscale folder created successfully!
Grayscale frames will be saved in: C:\Users\Mavith\OneDrive\Desktop\ip_project\Cross_marking_removel\pipeline\frames\grayscale
 Grayscale Conversion Completed!
 Total Frames Converted: 31
 Saved in: C:\Users\Mavith\OneDrive\Desktop\ip_project\Cross_marking_removel\pipeline\frames\grayscale
Enhanced folder created!
Enhanced images will be saved in: C:\Users\Mavith\OneDrive\Desktop\ip_project\Cross_marking_removel\pipeline\frames\enhanced
Median folder created!
Median results: C:\Users\Mavith\OneDrive\Desktop\ip_project\Cross_marking_removel\pipeline\frames\enhanced\median
CLAHE folde

,Frame,Mean Before,Mean After,Std Before,Std After,Entropy Before,Entropy After,PSNR
0,frame_0017.jpg,121.633053,60.014779,56.389242,28.161493,7.695753,6.686818,27.625222
1,frame_0018.jpg,121.000113,59.881611,56.776628,28.375516,7.727883,6.712372,27.536163
2,frame_0019.jpg,121.051950,59.628440,57.803157,28.841740,7.789804,6.779137,27.515567
3,frame_0020.jpg,121.999213,60.283543,57.144290,28.565926,7.767194,6.752100,27.440442
